**Title:** LRR Case Study - Exploratory Analysis for Part 1<br>
**Created By:** Begimay Ahmatova<br>
**Created Date:** 2026-10-02<br>
**Description:** Deep dive into raw files for company & people data modeling.<br>

#### 1. Setup

In [37]:
## libraries
import pandas as pd
import duckdb as db
import os

In [38]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [39]:
con = db.connect("lrr.duckdb")

In [40]:
%sql con

In [41]:
%config SqlMagic.displaylimit = 10

#### 2. Raw Ingestion

In [42]:
%%sql
CREATE OR REPLACE TABLE raw_company AS 
SELECT * 
FROM read_csv_auto('company.csv')

Running query in 'DuckDBPyConnection'

Count
1024


In [43]:
%%sql
CREATE OR REPLACE TABLE raw_person AS
SELECT *
FROM read_csv_auto('person.csv')

Running query in 'DuckDBPyConnection'

Count
1870


In [44]:
%%sql
CREATE OR REPLACE TABLE raw_persontocompany AS
SELECT *
FROM read_csv_auto('persontocompany.csv')

Running query in 'DuckDBPyConnection'

Count
1983


#### 3. Research Notes

research notes:<br>
- phonenumber can be standardized from XXX-XXX-XXX and +XXXXXXXXXXX to +XXXXXXXXXXX
- lower & trim informalname to standardize for dupe check
- remove inc, llc, co, ltd, corp, trailing period or comma from name to standardize

### Company 

#### 4. Table Overview

In [45]:
%%sql
select * from raw_company


Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate
9700,"Medi-Code, LLC",Medi-Code,http://medi-code.com,medi-code.com,832-437-2029,Katy,Texas,77494,United States,2009,1,None,None,None,False,2026-09-15 08:38:00.056721
20074,"Marketware, Inc.",Marketware,https://marketware.com,marketware.com,+18019444230,Sandy,Utah,84094,United States,2014,4,1665037,Medsphere Systems Corporation,5900000,False,2026-07-16 07:55:38.110403
28462,VCNetwork,VCNetwork,https://vcnetwork.org,vcnetwork.org,None,San Francisco,California,94114,United States,1997,5,None,None,None,False,2026-09-11 18:20:57.795627
35837,"Complipoint, LLC",Complipoint,https://www.complipoint.com,complipoint.com,+18586732174,San Diego,California,92127,United States,2002,8,204227,Applications International Corp. (AIC),None,False,2025-11-26 15:28:03.279874
45311,"FlexiInternational Software, Inc.",FlexiInternational Software,https://www.flexi.com,flexi.com,+18003539492,Shelton,Connecticut,06484,United States,1992,3,None,None,None,False,2026-08-08 18:12:09.990011
48678,"Blackhawk Specialty Tools, LLC",Blackhawk,http://www.blackhawkst.com,blackhawkst.com,713-466-4200,Houston,Texas,77042,United States,2008,4,63706,"Expro International Group, Ltd.",None,False,2026-08-15 17:03:50.885743
67565,"DemandFarm, Inc.",DemandFarm,https://www.demandfarm.com,demandfarm.com,+14157353143,New York City,New York,10111,United States,2016,1,None,None,None,False,2026-07-03 06:30:59.343793
68230,"Inspirage, LLC",Inspirage,https://inspirage.com,inspirage.com,+18555174250,Bellevue,Washington,98004,United States,2007,4,486849,Accenture PLC,None,False,2025-04-23 19:32:32.773916
71013,"Sutoer Solutions, Inc.",Sutoer Solutions,https://sutoer.com,sutoer.com,+16303214702,Hinsdale,Illinois,60521,United States,1997,1,None,None,None,False,2026-07-07 00:17:50.155018
83110,"Implantech Associates, Inc.",Implantech,https://www.implantech.com,implantech.com,+18053399415,Ventura,California,93003,United States,1987,1,None,None,None,False,2026-03-08 00:17:59.621450


In [46]:
%%sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'raw_company'
ORDER BY ordinal_position

Running query in 'DuckDBPyConnection'

column_name,data_type
id,BIGINT
name,VARCHAR
informalname,VARCHAR
website,VARCHAR
domain,VARCHAR
phonenumber,VARCHAR
city,VARCHAR
state,VARCHAR
postalcode,VARCHAR
country,VARCHAR


In [47]:
%%sql
select
    count(*) as total_rows,
    count(distinct id) as unique_id,
    count(distinct name) as unique_name,
    count(distinct informalname) as unique_informalname,
    count(distinct website) as unique_website,
    count(distinct domain) as unique_domain,
    count(distinct phonenumber) as unique_phonenumber,
    count(distinct parentid) as unique_parentid,
    count(distinct parentcompany) as unique_parentcompany
from raw_company


Running query in 'DuckDBPyConnection'

total_rows,unique_id,unique_name,unique_informalname,unique_website,unique_domain,unique_phonenumber,unique_parentid,unique_parentcompany
1024,1024,1015,1004,1001,999,902,62,64


In [48]:
%%sql
select
    (select count(*) from raw_company where id is null) as null_id,
    (select count(*) from raw_company where name is null or name = '') as null_name,
    (select count(*) from raw_company where informalname is null or informalname = '') as null_informalname,
    (select count(*) from raw_company where website is null or website = '') as null_website,
    (select count(*) from raw_company where domain is null or domain = '') as null_domain,
    (select count(*) from raw_company where phonenumber is null or phonenumber = '') as null_phonenumber


Running query in 'DuckDBPyConnection'

null_id,null_name,null_informalname,null_website,null_domain,null_phonenumber
0,0,4,0,0,111


In [49]:
%%sql
select 
    min(cast(modifieddate as date)) as min_date,
    max(cast(modifieddate as date)) as max_date
from raw_company

Running query in 'DuckDBPyConnection'

min_date,max_date
2025-04-04,2026-09-26


In [50]:
%%sql
select 
    cast(modifieddate as date) as modified_date,
    count(*) as cnt
from raw_company
group by cast(modifieddate as date)
order by cnt desc
limit 10

Running query in 'DuckDBPyConnection'

modified_date,cnt
2025-04-04,74
2025-04-05,43
2025-05-14,13
2025-05-17,13
2026-08-31,11
2026-08-15,11
2026-09-22,11
2026-08-13,10
2026-09-14,10
2026-09-23,10


#### 5. Company Name

In [51]:
%%sql
select name, count(*) as cnt
from raw_company
group by name
having cnt > 1

Running query in 'DuckDBPyConnection'

name,cnt
Netafim USA,2
Hoard's Dairyman Farm Creamery,2
Celerity Risk,2
"Grillo's Pickles, Inc.",2
"The Shyft Group, Inc.",2
"Baudville, Inc.",2
"Pacifier Online Data Service dba Infinity Internet, Inc.",2
"Kenilworth Media, Inc.",2
"Inspirage, LLC",2


In [52]:
%%sql
select *
from raw_company
where name = 'Kenilworth Media, Inc.'

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate
1834687,"Kenilworth Media, Inc.",Kenilworth Media,https://kenilworth.com,kenilworth.com,+19057717333,Richmond Hill,Ontario,L4B 3H7,Canada,1997,8,359884,Forum Media Group GmbH,None,False,2025-05-14 12:07:04.632082
18189331,"Kenilworth Media, Inc.",Kenilworth Media,https://kenilworth.com,kenilworth.com,None,Richmond Hill,Ontario,L4B 4N4,Canada,1997,8,359884,FORUM MEDIA GROUP GMBH,None,False,2026-08-13 21:07:40.099777


In [53]:
%%sql
SELECT
    name,
    trim(regexp_replace(
        regexp_replace(name, '[.,\s]+$', ''),
        '(?i)[,\s]+(inc|llc|co|ltd|corp)\.?$',
        ''
    )) AS cleaned_name
FROM raw_company;

Running query in 'DuckDBPyConnection'

name,cleaned_name
"Medi-Code, LLC",Medi-Code
"Marketware, Inc.",Marketware
VCNetwork,VCNetwork
"Complipoint, LLC",Complipoint
"FlexiInternational Software, Inc.",FlexiInternational Software
"Blackhawk Specialty Tools, LLC",Blackhawk Specialty Tools
"DemandFarm, Inc.",DemandFarm
"Inspirage, LLC",Inspirage
"Sutoer Solutions, Inc.",Sutoer Solutions
"Implantech Associates, Inc.",Implantech Associates


In [54]:
%%sql
SELECT
    trim(regexp_replace(
        regexp_replace(name, '[.,\s]+$', ''),
        '(?i)[,\s]+(inc|llc|co|ltd|corp)\.?$',
        ''
    )) AS cleaned_name, count(*) as cnt
FROM raw_company
group by cleaned_name
having cnt > 1
# order by cnt desc

EXCEPT 

select name, count(*) as cnt
from raw_company
group by name
having cnt > 1
# order by cnt desc

Running query in 'DuckDBPyConnection'

cleaned_name,cnt
Kenilworth Media,2
Fox Innovation & Technologies,2
Branded Bull,2
Encore Labs,2
Operator Academy,2
Baudville,2
Plastic Components,2
Grillo's Pickles,2
Pacifier Online Data Service dba Infinity Internet,2
Inspirage,2


In [55]:
%%sql
    SELECT trim(regexp_replace(regexp_replace(name, '[.,\s]+$', ''),
                '(?i)[,\s]+(inc|llc|co|ltd|corp)\.?$','')) as cleaned_name,
            count(*) as cnt
    # name, count(*) as cnt
    FROM raw_company
    GROUP BY cleaned_name
    HAVING cnt > 1

Running query in 'DuckDBPyConnection'

cleaned_name,cnt
Baudville,2
Plastic Components,2
Encore Labs,2
Netafim USA,2
Hoard's Dairyman Farm Creamery,2
Celerity Risk,2
Inspirage,2
The Shyft Group,2
Kenilworth Media,2
Fox Innovation & Technologies,2


#### 6. Informal Name

In [56]:
%%sql
select informalname, count(*) as cnt
from raw_company
group by informalname
having cnt > 1
order by cnt desc

Running query in 'DuckDBPyConnection'

informalname,cnt
None,4
Grillo's Pickles,2
Operator Academy,2
Baudville,2
Infinity Internet,2
Inspirage,2
Netafim USA,2
Hoard's Dairyman Farm Creamery,2
Celerity Risk,2
MLI,2


In [57]:
%%sql
select lower(trim(informalname)) as normalized_informalname, count(*) as cnt
from raw_company
group by normalized_informalname
having cnt > 1
EXCEPT
select informalname, count(*) as cnt
from raw_company
group by informalname
having cnt > 1

Running query in 'DuckDBPyConnection'

normalized_informalname,cnt
grillo's pickles,2
inspirage,2
mli,2
celerity risk,2
netafim usa,2
infinity internet,2
gomo health,2
kenilworth media,2
neighborly,2
encore labs,2


In [58]:
%%sql
select *, lower(trim(informalname)) as normalized_informalname
from raw_company
where normalized_informalname = 'fox innovation & technologies'

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate,normalized_informalname
15597800,"Fox Innovation & Technologies, Inc.",Fox Innovation & Technologies,https://www.foxinnovation.com,foxinnovation.com,+17164894999,Olean,New York,14760,United States,2022,3,None,None,None,False,2026-07-09 07:31:54.642184,fox innovation & technologies
17198335,Fox Innovation & Technologies,FOX Innovation & Technologies,https://www.foxinnovation.com,foxinnovation.com,+18702637043,La Porte,Texas,77571,United States,1981,3,None,None,None,False,2025-05-09 01:42:26.994298,fox innovation & technologies


In [59]:
%%sql
select *
from raw_company
where informalname is NUll

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate
207459,Evicient LLC,None,http://www.evicient.com,evicient.com,630-657-6800,Atlanta,Georgia,30309,United States,2010,1,None,None,None,False,2026-09-07 09:09:25.607328
225687,The Shine Project,None,http://www.theshineproject.com,theshineproject.com,480-424-4444,Phoenix,Arizona,85013,United States,2011,5,None,None,None,False,2025-09-09 03:57:54.717160
263651,My Job Coaches,None,http://www.myjobcoaches.com,myjobcoaches.com,320-363-5414,Waite Park,Minnesota,56387,United States,2013,1,None,None,None,False,2025-04-04 16:42:32.494008
334552,Barosolutions Wellness and Weight Loss Institute,None,http://barosolutions.com,barosolutions.com,817-479-9332,Dallas,Texas,75204,United States,2006,1,None,None,None,False,2025-04-04 15:33:23.320380


#### 7. Website & Domain

In [60]:
%%sql
select website, count(*) as cnt
from raw_company
group by website
having cnt > 1
order by cnt desc

Running query in 'DuckDBPyConnection'

website,cnt
https://theshyftgroup.com,2
https://www.operatoracademy.org,2
https://www.standardhotels.com,2
https://www.plattsburgh.edu,2
https://kenilworth.com,2
https://jscpa.com,2
https://www.neighborly.com,2
https://tumeryk.com,2
https://www.baudville.com,2
https://paullewallen.com,2


In [61]:
%%sql
select *
from raw_company
where website = 'https://www.standardhotels.com'

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate
256079,"Standard International Management, LLC",Standard International Management,https://www.standardhotels.com,standardhotels.com,212-645-4646,New York City,New York,10003,United States,1999,8,668461,"Standard International, LLC",None,False,2025-05-20 08:47:34.538262
18282048,The Standard Hotels,The Standard Hotels,https://www.standardhotels.com,standardhotels.com,+12126454646,New York City,New York,10003,United States,1999,8,668461,"Standard International, LLC",None,False,2026-01-09 18:48:46.964360


In [62]:
%%sql
select domain, count(*) as cnt
from raw_company
group by domain
having cnt > 1
order by cnt desc

Running query in 'DuckDBPyConnection'

domain,cnt
mlieducation.org,2
plasticcomponents.com,2
shepherd.vet,2
netafimusa.com,2
grillos.com,2
encorelabs.com,2
kenilworth.com,2
standardhotels.com,2
theshyftgroup.com,2
gomohealth.com,2


#### 8. Country

In [63]:
%%sql
select
    country,
    count(*) as cnt
from raw_company
group by country
order by cnt desc

Running query in 'DuckDBPyConnection'

country,cnt
United States,937
Canada,87


#### 9. Phone Number

In [64]:
%%sql
select
    phonenumber,
    count(*) as cnt
from raw_company
group by phonenumber
having cnt > 1
order by cnt desc

Running query in 'DuckDBPyConnection'

phonenumber,cnt
None,111
+14806242599,3
+18007280888,2
+14067521040,2
+16093331693,2
+18552178437,2
+15175436400,2
+13133075408,2
+19205635551,2
480-624-2505,2


In [65]:
%%sql
select *
from raw_company
where phonenumber = '+14806242599'

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate
144487,Sky Homes,Sky Homes,https://www.skyhomesusa.com,skyhomesusa.com,+14806242599,Boston,Massachusetts,02215,United States,2013,1,None,None,100000,False,2025-10-08 14:24:23.252445
1200184,"FanBeat, Inc.",FanBeat,https://www.fanbeat.com,fanbeat.com,+14806242599,Atlanta,Georgia,30319,United States,2014,2,None,None,None,False,2025-04-04 19:01:29.178884
16131288,PrimeVault,PrimeVault,https://www.primevault.com,primevault.com,+14806242599,San Francisco,California,94103,United States,2022,2,None,None,None,False,2026-09-14 04:49:04.584107


In [66]:
%%sql
SELECT
    length(regexp_replace(phonenumber, '[^0-9]', '', 'g')) AS digit_count,
    COUNT(*) AS rows
FROM raw_company
WHERE phonenumber IS NOT NULL
GROUP BY digit_count
ORDER BY digit_count;

Running query in 'DuckDBPyConnection'

digit_count,rows
10,280
11,632
12,1


In [67]:
%%sql
WITH cleaned AS (
    SELECT
        phonenumber,
        regexp_replace(trim(phonenumber), '[^0-9]', '', 'g') AS digits
    FROM raw_company
)
SELECT phonenumber, digits
FROM cleaned
WHERE (length(digits) = 11 AND left(digits, 1) <> '1')
   OR length(digits) = 12;

Running query in 'DuckDBPyConnection'

phonenumber,digits
+448009876543,448009876543


In [68]:
%%sql
select *
from raw_company
where phonenumber = '+448009876543'

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate
15048922,BAINS DEVELOPMENTS LIMITED,BAINS DEVELOPMENTS LIMITED,http://bainsdevelopmentsltd.co.uk,bainsdevelopmentsltd.co.uk,+448009876543,Brooklyn,New York,11203,United States,2015,1,None,None,None,False,2025-04-05 10:34:27.897880


In [69]:
%%sql
WITH cleaned AS (
    SELECT
        *,
        starts_with(trim(coalesce(phonenumber, '')), '+') AS has_plus,
        regexp_replace(coalesce(phonenumber, ''), '[^0-9]', '', 'g') AS digits
    FROM raw_company
),
normalized AS (
    SELECT
        *,
        CASE
            WHEN has_plus THEN '+' || digits
            WHEN length(digits) = 10
                AND lower(trim(country)) IN ('united states', 'canada')
                THEN '+1' || digits
            ELSE digits
        END AS normalized_phone
    FROM cleaned
)
# SELECT normalized_phone, count(*) AS cnt
# FROM normalized
# GROUP BY normalized_phone
# HAVING count(*) > 1
# ORDER BY cnt DESC;
SELECT *
FROM normalized
WHERE normalized_phone = '+16266963086'
# select count(distinct normalized_phone)
# from normalized

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate,has_plus,digits,normalized_phone
1473059,"Encore Labs, LLC",Encore Labs,https://encorelabs.com,encorelabs.com,626-696-3086,Pasadena,California,91107,United States,2017,1,None,None,None,False,2025-05-08 08:42:32.029784,False,6266963086,+16266963086
17248042,Encore Labs,ENCORE LABS,https://encorelabs.com,encorelabs.com,+16266963086,Pasadena,California,91105,United States,2017,1,None,None,None,False,2026-09-16 01:31:05.295047,True,16266963086,+16266963086


#### 10. Cross-Field Duplicate Check

In [70]:
%%sql
WITH dupe_name AS (
    SELECT trim(regexp_replace(regexp_replace(name, '[.,\s]+$', ''),
                '(?i)[,\s]+(inc|llc|co|ltd|corp)\.?$','')) as cleaned_name,
            count(*) as cnt
    # name, count(*) as cnt
    FROM raw_company
    GROUP BY cleaned_name
    HAVING cnt > 1
),
dupe_informalname AS (
    SELECT informalname, count(*) as cnt
    FROM raw_company
    GROUP BY informalname
    HAVING cnt > 1
),
dupe_website AS (
    SELECT website, count(*) as cnt
    FROM raw_company
    GROUP BY website
    HAVING cnt > 1
),
dupe_domain AS (
    SELECT domain, count(*) as cnt
    FROM raw_company
    GROUP BY domain
    HAVING cnt > 1
),
cleanedname as (
    SELECT *,
        trim(regexp_replace(regexp_replace(name, '[.,\s]+$', ''),
                '(?i)[,\s]+(inc|llc|co|ltd|corp)\.?$','')) as cleaned_name 
    FROM raw_company
)

# select *
# from cleanedname
# where informalname in (select informalname from dupe_informalname)

# EXCEPT

# select *
# from cleanedname
# where cleaned_name in (select cleaned_name from dupe_name)

select *
from cleanedname
where domain in (select domain from dupe_domain)

except

select *
from cleanedname
where informalname in (select informalname from dupe_informalname)

# select *
# from cleanedname
# where website in (select website from dupe_website)


order by domain asc

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate,cleaned_name
1473059,"Encore Labs, LLC",Encore Labs,https://encorelabs.com,encorelabs.com,626-696-3086,Pasadena,California,91107,United States,2017,1,None,None,None,False,2025-05-08 08:42:32.029784,Encore Labs
17248042,Encore Labs,ENCORE LABS,https://encorelabs.com,encorelabs.com,+16266963086,Pasadena,California,91105,United States,2017,1,None,None,None,False,2026-09-16 01:31:05.295047,Encore Labs
15597800,"Fox Innovation & Technologies, Inc.",Fox Innovation & Technologies,https://www.foxinnovation.com,foxinnovation.com,+17164894999,Olean,New York,14760,United States,2022,3,None,None,None,False,2026-07-09 07:31:54.642184,Fox Innovation & Technologies
17198335,Fox Innovation & Technologies,FOX Innovation & Technologies,https://www.foxinnovation.com,foxinnovation.com,+18702637043,La Porte,Texas,77571,United States,1981,3,None,None,None,False,2025-05-09 01:42:26.994298,Fox Innovation & Technologies
16596538,"Jordahl & Sliter, PLLC",Jordahl & Sliter,https://jscpa.com,jscpa.com,+14067521040,Kalispell,Montana,59901,United States,1952,1,None,None,None,False,2025-04-05 12:35:22.628861,"Jordahl & Sliter, PLLC"
1847585,"Jordahl and Sliter, PLLC",Jordahl and Sliter,https://jscpa.com,jscpa.com,+14067521040,Kalispell,Montana,59901,United States,1952,1,None,None,None,False,2026-06-06 16:15:54.620598,"Jordahl and Sliter, PLLC"
2663458,Paul Lewallen,Paul Lewallen,https://paullewallen.com,paullewallen.com,858-255-1192,San Diego,California,92137,United States,2005,1,None,None,None,False,2025-05-15 18:17:32.270775,Paul Lewallen
18277169,Paul Lewallen Media,Paul Lewallen Media,https://paullewallen.com,paullewallen.com,+18582551192,San Diego,California,92137,United States,2005,1,None,None,None,False,2026-06-30 22:25:27.960143,Paul Lewallen Media
21741443,State University of New York at Plattsburgh,State University of New York at Plattsburgh,https://www.plattsburgh.edu,plattsburgh.edu,+15185643094,Plattsburgh,New York,12901,United States,1889,12,None,None,None,False,2026-07-04 15:10:59.248224,State University of New York at Plattsburgh
715325,State University of New York at Plattsburgh (SUNY Plattsburgh),SUNY Plattsburgh,https://www.plattsburgh.edu,plattsburgh.edu,518-564-2000,Plattsburgh,New York,12901,United States,1889,12,None,None,None,False,2025-04-04 16:03:59.357493,State University of New York at Plattsburgh (SUNY Plattsburgh)


### Person

In [71]:
%%sql
select *
from raw_person

Running query in 'DuckDBPyConnection'

id,firstname,middlename,lastname,linkedin,facebook,twitter,website,city,state,country,deleted,statecode,countrycode,modifieddate,loaddate
100001,Patricia,None,Scott,https://www.linkedin.com/in/patricia-scott-27,None,None,None,None,None,United States,False,None,US,2024-07-28,2026-09-01
100002,Alexander,None,Cooper,None,None,None,None,None,None,United States,False,None,US,2026-05-14,2026-09-01
100003,Karen,None,Ramirez,https://www.linkedin.com/in/karen-ramirez-53,None,None,None,None,None,United States,False,None,US,2024-07-28,2026-09-01
100004,Jonathan,None,Thomas,https://www.linkedin.com/in/jonathan-thomas-58,None,None,None,None,None,United States,False,None,US,2024-06-10,2026-09-01
100005,Stephen,None,Nguyen,https://www.linkedin.com/in/stephen-nguyen-89,None,None,None,None,None,United States,False,None,US,2026-01-10,2026-09-01
100006,Pamela,None,Harris,https://www.linkedin.com/in/pamela-harris-15,None,None,None,None,None,United States,False,None,US,2025-04-11,2026-09-01
100007,Andrew,None,Rodriguez,https://www.linkedin.com/in/andrew-rodriguez-82,None,None,None,None,None,United States,False,None,US,2025-10-06,2026-09-01
100008,Justin,None,Phillips,https://www.linkedin.com/in/justin-phillips-27,None,None,None,None,None,United States,False,None,US,2026-11-09,2026-09-01
100009,Amy,None,Lopez,https://www.linkedin.com/in/amy-lopez-24,None,None,None,None,None,United States,False,None,US,2024-11-09,2026-09-01
100010,Ryan,None,Smith,https://www.linkedin.com/in/ryan-smith-43,None,None,None,None,None,United States,False,None,US,2026-10-22,2026-09-01


In [72]:
%%sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'raw_person' f
ORDER BY ordinal_position

Running query in 'DuckDBPyConnection'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
Parser Error: syntax error at or near "f"

LINE 3: WHERE table_name = 'raw_person' f
                                        ^



In [ ]:
%%sql
select
    count(*) as total_rows,
    count(distinct id) as unique_id,
    count(distinct firstname) as unique_firstname,
    count(distinct middlename) as unique_middlename,
    count(distinct lastname) as unique_lastname,
    count(distinct linkedin) as unique_li
from raw_person


Running query in 'DuckDBPyConnection'

total_rows,unique_id,unique_firstname,unique_middlename,unique_lastname,unique_li
1870,1870,100,0,80,1578


In [ ]:
%%sql
select
    (select count(*) from raw_person where id is null) as null_id,
    (select count(*) from raw_person where firstname is null) as null_firstname,
    (select count(*) from raw_person where middlename is null) as null_middlename,
    (select count(*) from raw_person where lastname is null) as null_lastname,
    (select count(*) from raw_person where firstname || ' ' || lastname is null) as null_fullname,
    (select count(*) from raw_person where linkedin is null) as null_li,
    (select count(*) from raw_person where modifieddate is null) as null_mdate


Running query in 'DuckDBPyConnection'

null_id,null_firstname,null_middlename,null_lastname,null_fullname,null_li,null_mdate
0,0,1870,0,0,291,0


In [ ]:
%%sql
select 
    min(cast(modifieddate as date)) as min_mdate,
    max(cast(modifieddate as date)) as max_mdate,
    min(cast(loaddate as date)) as min_ldate,
    max(cast(loaddate as date)) as max_ldate
from raw_person

Running query in 'DuckDBPyConnection'

min_mdate,max_mdate,min_ldate,max_ldate
2024-01-02,2026-12-31,2026-09-01,2026-09-01


In [ ]:
%%sql
select 
    cast(modifieddate as date) as modified_date,
    count(*) as cnt
from raw_person
group by cast(modifieddate as date)
order by cnt desc
limit 10

Running query in 'DuckDBPyConnection'

modified_date,cnt
2025-09-15,6
2026-11-30,6
2024-07-28,6
2026-05-21,6
2026-09-19,6
2026-01-25,6
2024-04-20,5
2024-05-02,5
2024-03-16,5
2024-07-26,5


In [ ]:
%%sql
with grouped as (
    select linkedin, count(*) as cnt
    from raw_person
    group by linkedin
    having cnt > 1
)
select *
from raw_person
where linkedin in (select linkedin from grouped)

Running query in 'DuckDBPyConnection'

id,firstname,middlename,lastname,linkedin,facebook,twitter,website,city,state,country,deleted,statecode,countrycode,modifieddate,loaddate
100856,Sarah,None,Reed,https://www.linkedin.com/in/sarah-reed-38,None,None,None,None,None,United States,False,None,US,2025-01-20,2026-09-01
101396,Sarah,None,Reed,https://www.linkedin.com/in/sarah-reed-38,None,None,None,None,None,United States,False,None,US,2025-10-19,2026-09-01


In [ ]:
%%sql
with person as (
    select *,
        trim(firstname) as fname,
        trim(lastname) as lname,
        fname || ' ' || lname as fullname
    from raw_person
)
select count(*) from (
select fullname, count(*) as cnt
from person
where fullname in (select fullname from person)
group by fullname
having cnt > 1
)

Running query in 'DuckDBPyConnection'

count_star()
192


In [ ]:
%%sql
with person as (
    select *,
        trim(firstname) as fname,
        trim(lastname) as lname,
        fname || ' ' || lname as fullname
    from raw_person
),
grouped as (
select fullname, count(*) as cnt
from person
where fullname in (select fullname from person)
group by fullname
having cnt > 1
)
select *
from person
where fullname in (select fullname from grouped)
order by fullname asc

Running query in 'DuckDBPyConnection'

id,firstname,middlename,lastname,linkedin,facebook,twitter,website,city,state,country,deleted,statecode,countrycode,modifieddate,loaddate,fname,lname,fullname
100016,Alexander,None,Gonzalez,None,None,None,None,None,None,United States,False,None,US,2025-11-25,2026-09-01,Alexander,Gonzalez,Alexander Gonzalez
100195,Alexander,None,Gonzalez,None,None,None,None,None,None,United States,False,None,US,2026-04-22,2026-09-01,Alexander,Gonzalez,Alexander Gonzalez
101630,Alexander,None,King,https://www.linkedin.com/in/alexander-king-37,None,None,None,None,None,United States,False,None,US,2025-10-29,2026-09-01,Alexander,King,Alexander King
100077,Alexander,None,King,https://www.linkedin.com/in/alexander-king-91,None,None,None,None,None,United States,False,None,US,2026-01-30,2026-09-01,Alexander,King,Alexander King
100485,Alexander,None,Wilson,https://www.linkedin.com/in/alexander-wilson-76,None,None,None,None,None,United States,False,None,US,2024-06-23,2026-09-01,Alexander,Wilson,Alexander Wilson
100530,Alexander,None,Wilson,https://www.linkedin.com/in/alexander-wilson-12,None,None,None,None,None,United States,False,None,US,2024-01-12,2026-09-01,Alexander,Wilson,Alexander Wilson
100828,Amanda,None,Peterson,None,None,None,None,None,None,United States,False,None,US,2024-04-10,2026-09-01,Amanda,Peterson,Amanda Peterson
101379,Amanda,None,Peterson,https://www.linkedin.com/in/amanda-peterson-81,None,None,None,None,None,United States,False,None,US,2025-01-09,2026-09-01,Amanda,Peterson,Amanda Peterson
100948,Amanda,None,Ward,https://www.linkedin.com/in/amanda-ward-39,None,None,None,None,None,United States,False,None,US,2025-10-10,2026-09-01,Amanda,Ward,Amanda Ward
101207,Amanda,None,Ward,https://www.linkedin.com/in/amanda-ward-91,None,None,None,None,None,United States,False,None,US,2025-05-20,2026-09-01,Amanda,Ward,Amanda Ward


In [ ]:
%%sql
with person as (
    select *,
        trim(firstname) as fname,
        trim(lastname) as lname,
        fname || ' ' || lname as fullname
    from raw_person
)
select count(*)
from (
select *
from person
qualify row_number() over (partition by fullname order by modifieddate desc) = 1
order by fullname asc
)

Running query in 'DuckDBPyConnection'

count_star()
1665


In [ ]:
%%sql
with person as (
    select *,
        trim(firstname) as fname,
        trim(lastname) as lname,
        fname || ' ' || lname as fullname
    from raw_person
)
select *
from person
qualify row_number() over (partition by fullname order by modifieddate desc) = 1
order by fullname asc

Running query in 'DuckDBPyConnection'

id,firstname,middlename,lastname,linkedin,facebook,twitter,website,city,state,country,deleted,statecode,countrycode,modifieddate,loaddate,fname,lname,fullname
101390,Alexander,None,Brown,https://www.linkedin.com/in/alexander-brown-53,None,None,None,None,None,United States,False,None,US,2026-12-22,2026-09-01,Alexander,Brown,Alexander Brown
100002,Alexander,None,Cooper,None,None,None,None,None,None,United States,False,None,US,2026-05-14,2026-09-01,Alexander,Cooper,Alexander Cooper
101546,Alexander,None,Cruz,https://www.linkedin.com/in/alexander-cruz-10,None,None,None,None,None,United States,False,None,US,2026-08-23,2026-09-01,Alexander,Cruz,Alexander Cruz
100195,Alexander,None,Gonzalez,None,None,None,None,None,None,United States,False,None,US,2026-04-22,2026-09-01,Alexander,Gonzalez,Alexander Gonzalez
100251,Alexander,None,Gutierrez,https://www.linkedin.com/in/alexander-gutierrez-60,None,None,None,None,None,United States,False,None,US,2024-07-28,2026-09-01,Alexander,Gutierrez,Alexander Gutierrez
101454,Alexander,None,Hall,https://www.linkedin.com/in/alexander-hall-96,None,None,None,None,None,United States,False,None,US,2026-04-05,2026-09-01,Alexander,Hall,Alexander Hall
100188,Alexander,None,Hill,https://www.linkedin.com/in/alexander-hill-85,None,None,None,None,None,United States,False,None,US,2024-06-22,2026-09-01,Alexander,Hill,Alexander Hill
101712,Alexander,None,Kelly,None,None,None,None,None,None,United States,False,None,US,2026-05-27,2026-09-01,Alexander,Kelly,Alexander Kelly
100077,Alexander,None,King,https://www.linkedin.com/in/alexander-king-91,None,None,None,None,None,United States,False,None,US,2026-01-30,2026-09-01,Alexander,King,Alexander King
100484,Alexander,None,Mitchell,https://www.linkedin.com/in/alexander-mitchell-96,None,None,None,None,None,United States,False,None,US,2026-10-03,2026-09-01,Alexander,Mitchell,Alexander Mitchell


### PersonToCompany

In [ ]:
%%sql
select *
from raw_persontocompany

Running query in 'DuckDBPyConnection'

id,personid,companyid,startdate,enddate,email,emailconfirmed,phone,title,role,deleted,modifieddate,loaddate
500001,100001,9700,2017-08-10,None,None,False,None,Chief Executive Officer,1,False,2024-02-24,2026-09-01
500002,100002,9700,2024-01-29,None,alexander.cooper@medi-code.com,True,(425) 659-5557,Chief Technology Officer,2,False,2024-11-22,2026-09-01
500003,100003,20074,2024-07-03,None,None,False,None,Chief Executive Officer,1,False,2025-06-25,2026-09-01
500004,100004,28462,2018-02-12,None,jonathan.thomas@vcnetwork.org,True,(438) 303-7227,Chief Executive Officer,1,False,2024-11-29,2026-09-01
500005,100005,28462,2021-10-31,None,stephen.nguyen@vcnetwork.org,True,(899) 863-2169,Chief Financial Officer,2,False,2025-05-16,2026-09-01
500006,100006,28462,2022-09-05,None,pamela.harris@vcnetwork.org,True,(904) 770-4598,Director of Sales,3,False,2024-04-24,2026-09-01
500007,100007,45311,2017-12-20,2025-05-10,andrew.rodriguez@flexi.com,True,(858) 669-3340,Chief Executive Officer,1,False,2025-06-22,2026-09-01
500008,100008,48678,2019-04-26,None,justin.phillips@blackhawkst.com,True,(632) 810-2040,Chief Executive Officer,1,False,2026-12-19,2026-09-01
500009,100009,48678,2023-01-31,2026-01-04,amy.lopez@blackhawkst.com,True,None,Managing Partner,1,False,2025-11-27,2026-09-01
500010,100010,67565,2023-07-21,2025-12-22,ryan.smith@demandfarm.com,True,(403) 356-7126,Chief Executive Officer,1,False,2025-10-25,2026-09-01


In [ ]:
%%sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'raw_persontocompany'
ORDER BY ordinal_position

Running query in 'DuckDBPyConnection'

column_name,data_type
id,BIGINT
personid,BIGINT
companyid,BIGINT
startdate,DATE
enddate,DATE
email,VARCHAR
emailconfirmed,BOOLEAN
phone,VARCHAR
title,VARCHAR
role,BIGINT


In [ ]:
%%sql
select
    count(*) as total_rows,
    count(distinct id) as unique_id,
    count(distinct personid) as unique_personid,
    count(distinct companyid) as unique_companyid,
    count(distinct email) as unique_email,
    count(distinct phone) as unique_phone
from raw_persontocompany

Running query in 'DuckDBPyConnection'

total_rows,unique_id,unique_personid,unique_companyid,unique_email,unique_phone
1983,1983,1870,902,1631,1109


In [ ]:
%%sql
select
    (select count(*) from raw_persontocompany where id is null) as null_id,
    (select count(*) from raw_persontocompany where personid is null) as null_personid,
    (select count(*) from raw_persontocompany where companyid is null) as null_companyid,
    (select count(*) from raw_persontocompany where email is null) as null_email,
    (select count(*) from raw_persontocompany where phone is null) as null_phone


Running query in 'DuckDBPyConnection'

null_id,null_personid,null_companyid,null_email,null_phone
0,0,0,253,815


In [ ]:
%%sql
select 
    min(cast(modifieddate as date)) as min_mdate,
    max(cast(modifieddate as date)) as max_mdate,
    min(cast(loaddate as date)) as min_ldate,
    max(cast(loaddate as date)) as max_ldate
from raw_persontocompany

Running query in 'DuckDBPyConnection'

min_mdate,max_mdate,min_ldate,max_ldate
2024-01-01,2026-12-30,2026-09-01,2026-09-01


In [ ]:
%%sql
with emails as (
    select email, count(*) as cnt
    from raw_persontocompany
    group by email
    having cnt > 1
)
select *
from raw_persontocompany
where email in (select email from emails)
order by email asc


Running query in 'DuckDBPyConnection'

id,personid,companyid,startdate,enddate,email,emailconfirmed,phone,title,role,deleted,modifieddate,loaddate
501811,101712,16292129,2023-12-19,None,alexander.kelly@alliedemployerservices.com,True,None,Chief Executive Officer,1,False,2026-07-06,2026-09-01
501812,101712,16292129,2024-07-28,None,alexander.kelly@alliedemployerservices.com,False,None,Chief Executive Officer,1,False,2026-08-12,2026-09-01
500597,100570,1256969,2020-07-11,None,alexander.thomas@dreicor.com,True,None,Office Manager,3,False,2025-03-04,2026-09-01
500598,100570,1256969,2022-03-22,None,alexander.thomas@dreicor.com,False,None,Office Manager,3,False,2024-08-15,2026-09-01
500206,100199,459018,2017-10-22,2024-06-14,amy.wright@newyorkcares.org,True,(480) 295-6905,VP of Operations,3,False,2024-12-04,2026-09-01
500207,100199,459018,2018-09-18,None,amy.wright@newyorkcares.org,False,(480) 295-6905,VP of Operations,3,False,2025-04-24,2026-09-01
501759,101664,16028569,2023-04-23,2023-03-01,andrew.johnson@material-methods.com,False,None,Chief Executive Officer,1,False,2024-10-04,2026-09-01
501760,101664,16028569,2024-06-21,None,andrew.johnson@material-methods.com,False,None,Chief Executive Officer,1,False,2025-11-25,2026-09-01
500216,100207,478655,2018-06-24,None,angela.clark@locusview.com,True,(865) 897-2083,Chief Executive Officer,1,False,2025-05-03,2026-09-01
500217,100207,478655,2018-06-26,None,angela.clark@locusview.com,False,(865) 897-2083,Chief Executive Officer,1,False,2026-06-08,2026-09-01


id can differ for same personid and companyid

In [ ]:
%%sql
select count(*)
from (
select 
    p.id,
    p.firstname || ' ' || p.lastname as fullname,
    pc.personid, 
    pc.companyid,
    pc.email,
    c.id as compid,
    c.normalized_name as compname
from raw_person p
left join deduped_ptc pc on p.id = pc.personid
left join deduped_company c on pc.companyid = c.id
where c.id is null
)

Running query in 'DuckDBPyConnection'

count_star()
26


In [ ]:
%%sql
with grouped as (
select domain, count(*) as cnt
from raw_company
group by domain
having cnt > 1
)
select *
from raw_company
where domain in (select domain from grouped)
order by domain asc, modifieddate desc

Running query in 'DuckDBPyConnection'

id,name,informalname,website,domain,phonenumber,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modifieddate
18210854,Aspire Software (Canada),Aspire Software,https://www.aspiresoftware.com,aspiresoftware.com,+18132884900,Montreal,Quebec,H4T 1A5,Canada,2016,1,None,None,None,False,2026-06-19 18:23:34.221083
8188826,Aspire Software,Aspire Software,https://www.aspiresoftware.com,aspiresoftware.com,+15143167647,Saint-Laurent-d'Orleans,Quebec,H4T 1A5,Canada,2016,13,None,None,None,False,2025-06-04 01:34:08.792472
18138968,"Baudville, Inc.",Baudville,https://www.baudville.com,baudville.com,+18007280888,Grand Rapids,Michigan,49512,United States,1983,1,None,None,None,False,2026-05-09 19:22:28.842281
737601,"Baudville, Inc.",Baudville,https://www.baudville.com,baudville.com,+18007280888,Grand Rapids,Michigan,49512,United States,1983,1,None,None,None,False,2025-05-26 21:33:06.097520
11389514,"Branded Bull, Inc.",Branded Bull,https://www.brandedbull.com,brandedbull.com,+13133075408,Commerce,Michigan,48390,United States,2021,1,None,None,None,False,2026-09-11 23:51:09.884853
16395121,"Branded Bull, LLC",Branded Bull,https://www.brandedbull.com,brandedbull.com,+13133075408,Temecula,California,92590,United States,2008,1,None,None,None,False,2025-05-02 19:11:10.461393
18218540,Celerity Risk,Celerity Risk,https://celerityrisk.com,celerityrisk.com,None,Atlanta,Georgia,30326,United States,2025,1,None,None,None,False,2025-10-06 05:16:39.128391
18137108,Celerity Risk,Celerity Risk,https://celerityrisk.com,celerityrisk.com,None,Atlanta,Georgia,30326,United States,2025,1,None,None,None,False,2025-06-17 23:17:43.691338
17248042,Encore Labs,ENCORE LABS,https://encorelabs.com,encorelabs.com,+16266963086,Pasadena,California,91105,United States,2017,1,None,None,None,False,2026-09-16 01:31:05.295047
1473059,"Encore Labs, LLC",Encore Labs,https://encorelabs.com,encorelabs.com,626-696-3086,Pasadena,California,91107,United States,2017,1,None,None,None,False,2025-05-08 08:42:32.029784


In [73]:
%%sql
with grouped as (
    select personid, count(*) as cnt
    from raw_persontocompany
    group by personid
    having cnt > 1
)
select *
from raw_persontocompany
where personid in (select personid from grouped)
order by personid, modifieddate desc


Running query in 'DuckDBPyConnection'

id,personid,companyid,startdate,enddate,email,emailconfirmed,phone,title,role,deleted,modifieddate,loaddate
500024,100023,103855,2024-09-24,None,katherine.ortiz@arrow-precision.com,False,(717) 700-5114,Chief Executive Officer,1,False,2025-06-20,2026-09-01
500023,100023,103855,2024-06-27,2026-05-17,katherine.ortiz@arrow-precision.com,True,(717) 700-5114,Chief Executive Officer,1,False,2024-03-31,2026-09-01
500030,100028,105516,2018-06-08,None,timothy.young@leisurelink.com,False,(749) 539-1452,General Manager,3,False,2025-12-08,2026-09-01
500029,100028,105516,2016-04-17,None,timothy.young@leisurelink.com,False,(749) 539-1452,General Manager,3,False,2025-06-27,2026-09-01
500073,100071,201053,2019-04-12,None,christopher.young@micromaterials-inc.com,True,None,Director of Sales,3,False,2025-08-27,2026-09-01
500074,100071,201053,2021-03-21,None,christopher.young@micromaterials-inc.com,False,None,Director of Sales,3,False,2025-07-16,2026-09-01
500133,100129,306909,2022-02-25,None,ashley.phillips@rsgweb.com,False,(308) 526-4960,Owner,1,False,2026-05-06,2026-09-01
500132,100129,306909,2020-03-25,2025-01-15,ashley.phillips@rsgweb.com,False,(308) 526-4960,Owner,1,False,2024-07-17,2026-09-01
500150,100145,347922,2017-11-21,None,susan.nelson@bonehamusa.com,False,None,Chief Executive Officer,1,False,2025-08-27,2026-09-01
500149,100145,347922,2016-10-27,None,susan.nelson@bonehamusa.com,True,None,Chief Executive Officer,1,False,2025-01-16,2026-09-01


In [ ]:
%%sql
CREATE OR REPLACE TABLE deduped_ptc AS (
    select *
    from raw_persontocompany
    qualify row_number() over (partition by personid order by modifieddate desc) = 1
)

Running query in 'DuckDBPyConnection'

Count
1870


In [74]:
con.close()